# 06 — Chaos Engineering & Fault Injection

**Pipeline:** DistributedTrainingSimulator — node failure, gradient corruption, slow-node, checkpoint/recovery

```
Distributed Training Simulation
        │
        ├── Scenario A: Node failure (random node dies mid-step)
        │       └── recovery: checkpoint reload + gang restart
        │
        ├── Scenario B: Gradient corruption (NaN injection)
        │       └── recovery: gradient clipping + skip step
        │
        ├── Scenario C: Slow node (stragglers in AllReduce)
        │       └── recovery: timeout + node exclusion
        │
        └── Monte Carlo: job survival probability vs fleet size
                    │
                    ▼
             Reliability curve plot
```

**References:**
- Check-N-Run: Eisenman et al. (2022) https://arxiv.org/abs/2401.06172
- Bamboo: Yang et al. (2023) https://arxiv.org/abs/2303.02399

> **CPU-friendly:** This is a pure simulation — no GPU required.

In [ ]:
from __future__ import annotations

import os
import logging
import random

import wandb

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

CFG = {
    "num_nodes": 64,
    "failure_rate_per_hour": 0.01,
    "training_hours": 72,
    "checkpoint_interval_min": 15,
    "monte_carlo_trials": 1000,
    "scenarios": ["node_failure", "gradient_corruption", "slow_node"],
}

if not os.getenv("WANDB_DISABLED"):
    wandb.init(project="ai-transformer-research-hub", config=CFG, job_type="chaos-test")

In [ ]:
# TODO (Phase 3): implement DistributedTrainingSimulator
# Check-N-Run: Eisenman et al. (2022) https://arxiv.org/abs/2401.06172
# Bamboo: Yang et al. (2023) https://arxiv.org/abs/2303.02399

class DistributedTrainingSimulator:
    """Stub simulator for distributed training fault injection.

    TODO (Phase 3): implement full simulation logic.
    """

    def __init__(self, num_nodes: int, failure_rate: float, checkpoint_interval_min: float) -> None:
        self.num_nodes = num_nodes
        self.failure_rate = failure_rate
        self.checkpoint_interval_min = checkpoint_interval_min
        log.info(
            "[STUB] Simulator: %d nodes, failure_rate=%.4f, ckpt_interval=%d min",
            num_nodes, failure_rate, checkpoint_interval_min,
        )

    def run_scenario(self, scenario: str, training_hours: float) -> dict:
        """TODO (Phase 3): simulate scenario and return metrics."""
        log.info("[STUB] Running scenario: %s for %.1f hours", scenario, training_hours)
        return {"scenario": scenario, "survived": True, "lost_progress_min": 0.0}

    def monte_carlo_survival(self, trials: int) -> float:
        """TODO (Phase 3): Monte Carlo job survival probability."""
        log.info("[STUB] Running %d Monte Carlo trials", trials)
        return 0.0


sim = DistributedTrainingSimulator(
    num_nodes=CFG["num_nodes"],
    failure_rate=CFG["failure_rate_per_hour"],
    checkpoint_interval_min=CFG["checkpoint_interval_min"],
)

In [ ]:
# Run all scenarios
results = []
for scenario in CFG["scenarios"]:
    result = sim.run_scenario(scenario, CFG["training_hours"])
    results.append(result)
    if not os.getenv("WANDB_DISABLED"):
        wandb.log({f"{scenario}/survived": result["survived"]})

survival_prob = sim.monte_carlo_survival(CFG["monte_carlo_trials"])
log.info("[STUB] Estimated job survival probability: %.2f%%", survival_prob * 100)

# TODO (Phase 3): plot reliability curve (survival probability vs fleet size)

if not os.getenv("WANDB_DISABLED"):
    wandb.log({"survival_probability": survival_prob})
    wandb.finish()